In [6]:
!pip install ultralytics



[notice] A new release of pip available: 22.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:

import torch
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



CUDA available: True
Torch CUDA: 12.1
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [4]:
import torch, sys, subprocess, pkgutil

print("CUDA available:", torch.cuda.is_available(), "| Torch CUDA:", torch.version.cuda)
if not pkgutil.find_loader("ultralytics"):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics"])


CUDA available: True | Torch CUDA: 12.1


In [5]:
from pathlib import Path

base = Path(r"C:\Users\Tuf\Downloads\AI Backend\dataset_trans")  # <-- change if needed
yaml_path = base / "data.yaml"

data_yaml = f"""\
path: {base.as_posix()}
train: images/train
val: images/valid        # change to images/valid if that's your folder
test: images/test

nc: 5
names: ['Loose Joint -potential', 'Full wire overload', 'Loose Joint -Faulty', 'Point Overload Faulty', 'normal']
"""

yaml_path.write_text(data_yaml)
print("Wrote:", yaml_path)
print(yaml_path.read_text())


Wrote: C:\Users\Tuf\Downloads\AI Backend\dataset_trans\data.yaml
path: C:/Users/Tuf/Downloads/AI Backend/dataset_trans
train: images/train
val: images/valid        # change to images/valid if that's your folder
test: images/test

nc: 5
names: ['Loose Joint -potential', 'Full wire overload', 'Loose Joint -Faulty', 'Point Overload Faulty', 'normal']



In [6]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")           # or "yolo11n.pt" if VRAM is tight

results = model.train(
    data=str(yaml_path),
    device=0,              # use CUDA GPU 0
    imgsz=640,
    epochs=100,
    batch=16,
    workers=0,             # safer on Windows notebooks
    mosaic=0,
    hsv_h=0, hsv_s=0.10, hsv_v=0.10,
    verbose=True,
)

print("Run dir:", results.save_dir)   # e.g., runs/detect/train


Ultralytics 8.3.204  Python-3.11.1 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Tuf\Downloads\AI Backend\dataset_trans\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0, hsv_s=0.1, hsv_v=0.1, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patie

In [7]:
from ultralytics import YOLO
from pathlib import Path

# Locate your best weights from the last run
last_run = Path("runs/detect/train")
best = last_run/"weights"/"best.pt"

model = YOLO(str(best))
val_results = model.val(
    data=str(Path(r"C:\Users\Tuf\Downloads\AI Backend\dataset_trans\data.yaml")),
    split="test",            # evaluate on test split
    device=0,                # GPU
    save_json=True,
    plots=True               # saves confusion_matrix.png, PR_curve.png, results.png
)

print("Metrics dir:", val_results.save_dir)


Ultralytics 8.3.204  Python-3.11.1 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11s summary (fused): 100 layers, 9,414,735 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 8.15.2 MB/s, size: 67.6 KB)
val: Scanning C:\Users\Tuf\Downloads\AI Backend\dataset_trans\labels\test... 10 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 10/10 375.0it/s 0.0s
val: New cache created: C:\Users\Tuf\Downloads\AI Backend\dataset_trans\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 0.2it/s 4.4s
                   all         10         16      0.625      0.583      0.524      0.246
    Full wire overload          7          9      0.622      0.549      0.493      0.325
   Loose Joint -Faulty          4          5       0.89        0.2      0.415      0.108
                normal          2          2      0.361          1      0.663      0.305
Speed: 3.8

In [8]:
from ultralytics import YOLO
from pathlib import Path
import glob, json, csv

NAMES = [
    "Loose Joint -potential",
    "Full wire overload",
    "Loose Joint -Faulty",
    "Point Overload Faulty",
    "normal"
]
FAULTY = {2,3}
POTENTIAL = {0,1}

weights = Path("runs/detect/train/weights/best.pt")
data_root = Path(r"C:\Users\Tuf\Downloads\AI Backend\dataset_trans")
split = "test"

m = YOLO(str(weights))
images = sorted(glob.glob(str(data_root / "images" / split / "*.*")))

rows, jl = [], []
for img in images:
    r = m.predict(img, device=0, half=True, imgsz=640, verbose=False)[0]
    per_max = {i:0.0 for i in range(len(NAMES))}
    dets = []
    if len(r.boxes):
        cls = r.boxes.cls.cpu().numpy()
        conf = r.boxes.conf.cpu().numpy()
        for c,cf in zip(cls, conf):
            c = int(c); cf = float(cf)
            per_max[c] = max(per_max[c], cf)
            dets.append({"class_id": c, "class_name": NAMES[c], "conf": cf})

    S_faulty = max([per_max[i] for i in FAULTY], default=0.0)
    S_pot    = max([per_max[i] for i in POTENTIAL], default=0.0)

    grade = "faulty" if S_faulty>0 else ("potentially faulty" if S_pot>0 else "normal")
    score = max(S_faulty, 0.7*S_pot)  # simple monotonic score (we’ll replace with rules later)

    row = {"image": img, "grade": grade, "anomaly_score": round(score,4)}
    for i,n in enumerate(NAMES):
        row[f"max_{n}"] = round(per_max[i],4)
    row["num_detections"] = len(dets)
    rows.append(row)
    jl.append({"image": img, "grade": grade, "anomaly_score": score, "detections": dets})

out_dir = Path("runs/detect/summary_yolo_only"); out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / f"{split}_summary.csv"
jsonl_path = out_dir / f"{split}_summary.jsonl"

with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
with open(jsonl_path, "w") as f:
    for r in jl: f.write(json.dumps(r)+"\n")

print("Saved:", csv_path, "and", jsonl_path)


Saved: runs\detect\summary_yolo_only\test_summary.csv and runs\detect\summary_yolo_only\test_summary.jsonl


In [9]:
import json, os
cfg = {
    "yolo_min_conf": 0.20,                 # accept YOLO boxes above this
    "shape": {
        "min_area_frac": 0.001,            # ignore tiny blobs globally
        "loose_area_frac": 0.10,           # area fraction -> loose joint (faulty)
        "wire_aspect_min": 2.2,            # long/skinny -> wire
        "wire_area_max": 0.25              # keep wire class only if thinish
    },
    "color": {
        "sat_min": 0.35,                   # warm pixel saturation
        "val_min": 0.50,                   # warm pixel brightness
        "dv_min": 0.12,                    # baseline contrast (cand - baseline local)
        "dl_min": 0.08                     # local contrast (cand - local median)
    },
    "decision": {
        "rule_prob_thr": 0.40              # global rule probability cut (used later)
    }
}
os.makedirs("cfg", exist_ok=True)
with open("cfg/config_global.json","w") as f: json.dump(cfg, f, indent=2)
print("Wrote cfg/config_global.json")
print(json.dumps(cfg, indent=2))


Wrote cfg/config_global.json
{
  "yolo_min_conf": 0.2,
  "shape": {
    "min_area_frac": 0.001,
    "loose_area_frac": 0.1,
    "wire_aspect_min": 2.2,
    "wire_area_max": 0.25
  },
  "color": {
    "sat_min": 0.35,
    "val_min": 0.5,
    "dv_min": 0.12,
    "dl_min": 0.08
  },
  "decision": {
    "rule_prob_thr": 0.4
  }
}


Maximize recall (catch everything):

- yolo_min_conf: 0.15–0.20
- color.dl_min: 0.06–0.08
- color.dv_min: 0.10–0.12
- color_hot.dv_min_hot: 0.16–0.18
- shape.min_area_frac: 0.0007–0.001

Reduce false positives on normal images:

- Increase color.dv_min to 0.14–0.18
- Increase color_hot.dv_min_hot to 0.20–0.24
- Increase color.sat_min to 0.40
- Increase shape.min_area_frac to 0.0015–0.002
- Increase yolo_min_conf to 0.25

Wire getting mislabeled as loose joint:

- Raise shape.wire_aspect_min (e.g., 2.6)
- Lower shape.loose_area_frac only if big compact blobs are truly joints.
- Small point hotspots missed:
- Lower yolo_min_conf (0.15)
- Lower shape.min_area_frac a touch (0.0008)
- Lower color.dl_min slightly (0.06)

In [23]:
from ultralytics import YOLO
import numpy as np, cv2, json, glob, csv
from pathlib import Path
from math import exp

# ==== Config / paths ====
DATA_ROOT = Path(r"C:\Users\Tuf\Downloads\AI Backend\dataset_trans")
SPLIT = "test"
WEIGHTS = Path("runs/detect/train/weights/best.pt")
CFG = json.load(open("cfg/config_global.json"))

# Optional: map transformer -> baseline image path (leave empty to skip baseline compare)
# Example:
# BASELINES = {"T1": r"C:\...\baselines\T1_normal_001.jpg", ...}
BASELINES = {}

# ==== Utils ====
def load_rgb(path):
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None: raise FileNotFoundError(path)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def normalize_palette(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    h,s,v = cv2.split(hsv)
    v = cv2.equalizeHist(v)
    hsvn = cv2.merge([h,s,v])
    return cv2.cvtColor(hsvn, cv2.COLOR_HSV2RGB)

def resize_to_match(a, b):
    H, W = a.shape[:2]
    return cv2.resize(b, (W, H), interpolation=cv2.INTER_AREA)

def register_affine(base_rgb, cand_rgb):
    b_gray = cv2.cvtColor(base_rgb, cv2.COLOR_RGB2GRAY)
    c_gray = cv2.cvtColor(cand_rgb, cv2.COLOR_RGB2GRAY)
    warp = np.eye(2,3, dtype=np.float32)
    try:
        _, warp = cv2.findTransformECC(
            b_gray, c_gray, warp, cv2.MOTION_AFFINE,
            (cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT, 100, 1e-6)
        )
        H,W = b_gray.shape
        cand_reg = cv2.warpAffine(cand_rgb, warp, (W,H), flags=cv2.INTER_LINEAR+cv2.WARP_INVERSE_MAP)
        return cand_reg
    except cv2.error:
        return cand_rgb  # fallback if ECC fails

def rgb_to_hsv01(img_rgb):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[...,0] /= 179.0
    hsv[...,1] /= 255.0
    hsv[...,2] /= 255.0
    return hsv

def local_bg_contrast(v_base, v_cand, k=31):
    # median filters for local background; k must be odd
    med_base = cv2.medianBlur((v_base*255).astype(np.uint8), k)/255.0
    med_cand = cv2.medianBlur((v_cand*255).astype(np.uint8), k)/255.0
    dv = (v_cand - med_base)   # baseline delta
    dl = (v_cand - med_cand)   # local delta
    return dv, dl

# classify a single box using thresholds
def classify_box(hsvC, vB, vC, xywh, cfg):
    x,y,w,h = map(int, xywh)
    H, W = hsvC.shape[:2]
    x = max(0, min(x, W-1)); y = max(0, min(y, H-1))
    w = max(1, min(w, W-x)); h = max(1, min(h, H-y))

    roi = hsvC[y:y+h, x:x+w]
    hC,sC,vC_roi = roi[...,0], roi[...,1], roi[...,2]

    warm_h = (hC <= 0.08) | (hC >= 0.95)                 # reddish band
    warm_s = sC >= cfg["color"]["sat_min"]
    warm_v = vC_roi >= cfg["color"]["val_min"]

    if vB is None:
        med_base = None
    else:
        vB_roi = vB[y:y+h, x:x+w]
        med_base = cv2.medianBlur((vB_roi*255).astype(np.uint8), 31)/255.0

    med_cand = cv2.medianBlur((vC_roi*255).astype(np.uint8), 31)/255.0

    if med_base is None:
        dl = (vC_roi - med_cand)
        contrast = (dl >= cfg["color"]["dl_min"])
    else:
        dv = (vC_roi - med_base)
        dl = (vC_roi - med_cand)
        contrast = (dv >= cfg["color"]["dv_min"]) | (dl >= cfg["color"]["dl_min"])

    warm = (warm_h & warm_s & warm_v & contrast).astype(np.uint8)
    area = int(warm.sum())
    area_frac = area / float(H*W)

    short = max(1, min(w,h)); long = max(w,h)
    aspect = long/short

    # decide class by thresholds
    if area_frac >= cfg["shape"]["loose_area_frac"]:
        rule_label = "loose_joint"           # faulty
    elif aspect >= cfg["shape"]["wire_aspect_min"] and area_frac < cfg["shape"]["wire_area_max"]:
        rule_label = "wire_overload"         # potentially faulty
    else:
        rule_label = "point_overload"        # faulty (small blob)

    # drop if not warm enough at all
    is_warm = warm.sum() >= max(4, int(cfg["shape"]["min_area_frac"] * H * W))
    return is_warm, rule_label, float(area_frac), float(aspect)

# reconcile YOLO class name with rule label (simple policy)
YOLO_NAMES = [
    "Loose Joint -potential",
    "Full wire overload",
    "Loose Joint -Faulty",
    "Point Overload Faulty",
    "normal"
]
FAULTY_NAMES = {"Loose Joint -Faulty", "Point Overload Faulty"}
POTENTIAL_NAMES = {"Loose Joint -potential", "Full wire overload"}

def reconcile(yolo_name, rule_label, area_frac, aspect, cfg):
    # Drop normals outright
    if yolo_name == "normal":
        return None

    # Upgrade potentials if rule deems it large (loose_area_frac)
    if yolo_name in POTENTIAL_NAMES and rule_label == "loose_joint":
        return "Loose Joint -Faulty"

    # If rule says wire but it's not elongated enough, map to point overload
    if yolo_name == "Full wire overload" and rule_label == "point_overload":
        return "Point Overload Faulty"

    # If YOLO already says faulty and rule agrees it's warm, keep faulty class
    if yolo_name in FAULTY_NAMES:
        return yolo_name

    # Default: map rule label to nearest YOLO class name
    return {
        "loose_joint": "Loose Joint -Faulty",
        "wire_overload": "Full wire overload",
        "point_overload": "Point Overload Faulty"
    }[rule_label]

def global_rule_probe(hsvC, vB, cfg):
    hC, sC, vC = hsvC[...,0], hsvC[...,1], hsvC[...,2]
    warm_h = (hC <= 0.17) | (hC >= 0.95)
    warm_s = sC >= cfg["color"]["sat_min"]
    warm_v = vC >= cfg["color"]["val_min"]

    if vB is None:
        med_base = None
    else:
        med_base = cv2.medianBlur((vB*255).astype(np.uint8), 31)/255.0
    med_cand = cv2.medianBlur((vC*255).astype(np.uint8), 31)/255.0

    if med_base is None:
        dl = (vC - med_cand)
        contrast = (dl >= cfg["color"]["dl_min"])
        dv_like = dl
    else:
        dv = (vC - med_base)
        dl = (vC - med_cand)
        contrast = (dv >= cfg["color"]["dv_min"]) | (dl >= cfg["color"]["dl_min"])
        dv_like = np.maximum(0.0, dv)

    warm = (warm_h & warm_s & warm_v & contrast).astype(np.uint8)
    warm_frac = float(warm.mean())
    dv95 = float(np.quantile(dv_like.ravel(), 0.95))

    score = dv95 + 2.0*warm_frac     # recall-first
    prob = 1.0/(1.0 + exp(-score))
    return float(prob), warm

def propose_boxes_from_warm(warm_mask, cfg):
    H, W = warm_mask.shape
    min_area = max(32, int(cfg["shape"]["min_area_frac"] * H * W))
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
    wm = cv2.morphologyEx(warm_mask, cv2.MORPH_OPEN, k)
    num, labels, stats, _ = cv2.connectedComponentsWithStats(wm, 8)
    boxes = []
    for i in range(1, num):
        x,y,w,h,area = map(int, stats[i])
        if area >= min_area:
            boxes.append((x,y,w,h))
    return boxes

# ==== Run refinement over the split ====
model = YOLO(str(WEIGHTS))

images = sorted(glob.glob(str(DATA_ROOT / "images" / SPLIT / "*.*")))
rows, jl = [], []

for img_path in images:
    p = Path(img_path)
    # detect TX id from filename prefix like T4_... (tweak if different)
    tx_id = p.name.split("_", 1)[0]  # e.g., "T4"
    baseline_path = BASELINES.get(tx_id)

    raw  = load_rgb(img_path)                 # NEW
    cand = normalize_palette(raw)    
             # normalized copy for rules
    vB = None
    if baseline_path and Path(baseline_path).exists():
        base = normalize_palette(load_rgb(baseline_path))
        cand = resize_to_match(base, cand)
        cand = register_affine(base, cand)
        vB = rgb_to_hsv01(base)[...,2]

    hsvC = rgb_to_hsv01(cand); vC = hsvC[...,2]

    r = model.predict(raw, device=0, half=True, imgsz=640, verbose=False)[0]

    refined = []
    if len(r.boxes):
        xyxy = r.boxes.xyxy.cpu().numpy()
        cls = r.boxes.cls.cpu().numpy()
        conf = r.boxes.conf.cpu().numpy()
        for (x1,y1,x2,y2), c, cf in zip(xyxy, cls, conf):
            name = YOLO_NAMES[int(c)]
            cf = float(cf)
            if cf < CFG["yolo_min_conf"]:
                continue
            x,y,w,h = int(x1),int(y1),int(x2-x1),int(y2-y1)

            is_warm, rule_label, area_frac, aspect = classify_box(hsvC, vB, vC, (x,y,w,h), CFG)
            if not is_warm:
                continue  # drop cold regions

            final_name = reconcile(name, rule_label, area_frac, aspect, CFG)
            if final_name is None:
                continue  # drop normals

            refined.append({
                "x": x, "y": y, "w": w, "h": h,
                "detConfidence": cf,
                "ruleLabel": rule_label,
                "finalClass": final_name,
                "areaFrac": area_frac,
                "aspect": aspect
            })

     # --- HYBRID FALLBACK: if YOLO found nothing but rules say it's hot, still flag ---
    rule_prob_thr = CFG.get("decision", {}).get("rule_prob_thr", 0.4)
    use_rule_fallback = CFG.get("decision", {}).get("use_rule_only_fallback", True)

    rule_prob, warm_mask = global_rule_probe(hsvC, vB, CFG)

    if (len(refined) == 0) and use_rule_fallback and (rule_prob >= rule_prob_thr):
        for (x, y, w, h) in propose_boxes_from_warm(warm_mask, CFG):
            is_warm, rule_label, area_frac, aspect = classify_box(hsvC, vB, vC, (x,y,w,h), CFG)
            if not is_warm:
                continue
            # Map rule label to your dataset class names
            if rule_label == "wire_overload":
                final_name = "Full wire overload"          # potential by rubric
            elif rule_label == "loose_joint":
                # without "hot" flag here we conservatively set potential; you can tighten later
                final_name = "Loose Joint -potential"
            else:  # point_overload
                final_name = "Point Overload Faulty"       # usually faulty even when small

            refined.append({
                "x": x, "y": y, "w": w, "h": h,
                "detConfidence": float(rule_prob),  # proxy confidence for fallback
                "ruleLabel": rule_label,
                "finalClass": final_name,
                "areaFrac": float(area_frac),
                "aspect": float(aspect)
            })
        
    # Image-level grade after refinement
    has_faulty = any(b["finalClass"] in {"Loose Joint -Faulty","Point Overload Faulty"} for b in refined)
    has_pot    = any(b["finalClass"] in {"Loose Joint -potential","Full wire overload"} for b in refined)
    grade = "faulty" if has_faulty else ("potentially faulty" if has_pot else "normal")

    score = 0.0
    if refined:
        # simple image score = max refined confidence, weighted
        weights = {"Loose Joint -Faulty":1.0, "Point Overload Faulty":1.0,
                   "Loose Joint -potential":0.7, "Full wire overload":0.7}
        score = max(weights[b["finalClass"]] * b["detConfidence"] for b in refined)

    rows.append({
        "image": str(p),
        "grade_refined": grade,
        "anomaly_score_refined": round(float(score), 4),
        "num_boxes_refined": len(refined),
        "rule_prob": round(float(rule_prob), 4)
    })
    jl.append({
        "image": str(p),
        "grade_refined": grade,
        "anomaly_score_refined": float(score),
        "boxes": refined,
        "rule_prob": float(rule_prob)
    })

out_dir = Path("runs/detect/summary_refined"); out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / f"{SPLIT}_refined.csv"
jsonl_path = out_dir / f"{SPLIT}_refined.jsonl"

with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
with open(jsonl_path, "w") as f:
    for r in jl: f.write(json.dumps(r)+"\n")

print("Saved refined:", csv_path, "and", jsonl_path)


Saved refined: runs\detect\summary_refined\test_refined.csv and runs\detect\summary_refined\test_refined.jsonl


In [ ]:
# # === One-image inference with hot/warm rules ===
# from ultralytics import YOLO
# from pathlib import Path
# import numpy as np, cv2, json

# # ---- EDIT THESE ----
# CANDIDATE_IMG = r"C:\Users\Tuf\Downloads\AI Backend\dataset_trans\images\test\T4_faulty_001_png.rf.fdb4096aeac90695ff86252b9734247f.jpg"
# BASELINE_IMG  = None   # or e.g. r"C:\...\baselines\T4_normal_001.jpg"
# WEIGHTS       = r"runs\detect\train\weights\best.pt"
# CFG_PATH      = r"cfg\config_global.json"
# SAVE_PATH     = r"runs\detect\preview\annotated_one.png"
# # --------------------

# # --- load config ---
# CFG = json.load(open(CFG_PATH))

# def cfgv(path, default):
#     """Nested get with default: cfgv('color.dv_min', 0.12)"""
#     d = CFG
#     for k in path.split("."):
#         if not isinstance(d, dict) or k not in d:
#             return default
#         d = d[k]
#     return d

# # --- utils ---
# def load_rgb(path):
#     bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
#     if bgr is None: raise FileNotFoundError(path)
#     return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

# def normalize_palette(rgb):
#     hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
#     h,s,v = cv2.split(hsv)
#     v = cv2.equalizeHist(v)  # contrast-normalize V only
#     return cv2.cvtColor(cv2.merge([h,s,v]), cv2.COLOR_HSV2RGB)

# def resize_to_match(a, b):
#     H, W = a.shape[:2]
#     return cv2.resize(b, (W, H), interpolation=cv2.INTER_AREA)

# def register_affine(base_rgb, cand_rgb):
#     b_gray = cv2.cvtColor(base_rgb, cv2.COLOR_RGB2GRAY)
#     c_gray = cv2.cvtColor(cand_rgb, cv2.COLOR_RGB2GRAY)
#     warp = np.eye(2,3, dtype=np.float32)
#     try:
#         _, warp = cv2.findTransformECC(
#             b_gray, c_gray, warp, cv2.MOTION_AFFINE,
#             (cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT, 100, 1e-6)
#         )
#         H,W = b_gray.shape
#         return cv2.warpAffine(cand_rgb, warp, (W,H), flags=cv2.INTER_LINEAR+cv2.WARP_INVERSE_MAP)
#     except cv2.error:
#         return cand_rgb

# def rgb_to_hsv01(img_rgb):
#     hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
#     hsv[...,0] /= 179.0  # H in [0..1]
#     hsv[...,1] /= 255.0  # S in [0..1]
#     hsv[...,2] /= 255.0  # V in [0..1]
#     return hsv

# # --- per-box classification with warm & hot masks ---
# def classify_box(hsvC, vB, xywh):
#     x,y,w,h = map(int, xywh)
#     H, W = hsvC.shape[:2]
#     x = max(0, min(x, W-1)); y = max(0, min(y, H-1))
#     w = max(1, min(w, W-x)); h = max(1, min(h, H-y))

#     roi = hsvC[y:y+h, x:x+w]
#     hC, sC, vC_roi = roi[...,0], roi[...,1], roi[...,2]

#     # Hue: red -> orange/yellow (wrap)
#     warm_h = (hC <= 0.17) | (hC >= 0.95)

#     # WARM gates (recall-friendly)
#     warm_s = sC >= cfgv("color.sat_min", 0.35)
#     warm_v = vC_roi >= cfgv("color.val_min", 0.50)

#     # contrasts vs baseline & local
#     if vB is None:
#         vB_roi = np.zeros_like(vC_roi)
#     else:
#         vB_roi = vB[y:y+h, x:x+w]
#     med_base = cv2.medianBlur((vB_roi*255).astype(np.uint8), 31)/255.0
#     med_cand = cv2.medianBlur((vC_roi*255).astype(np.uint8), 31)/255.0
#     dv = (vC_roi - med_base)
#     dl = (vC_roi - med_cand)

#     contrast_warm = (dv >= cfgv("color.dv_min", 0.12)) | (dl >= cfgv("color.dl_min", 0.08))
#     warm = (warm_h & warm_s & warm_v & contrast_warm)

#     # HOT gates (stricter)
#     hot_s = sC >= cfgv("color_hot.sat_min_hot", 0.45)
#     hot_v = vC_roi >= cfgv("color_hot.val_min_hot", 0.65)
#     hot_contrast = (dv >= cfgv("color_hot.dv_min_hot", 0.18)) | (dl >= cfgv("color.dl_min", 0.08))
#     hot = (warm_h & hot_s & hot_v & hot_contrast)

#     warm_pix = int(warm.sum())
#     hot_pix  = int(hot.sum())

#     min_pix = max(4, int(cfgv("shape.min_area_frac", 0.001) * H * W))
#     ok_warm = warm_pix >= min_pix
#     ok_hot  = hot_pix  >= min_pix

#     # features (on warm geometry)
#     area_frac_warm = warm_pix / float(H*W)
#     short = max(1, min(w,h)); long = max(w,h)
#     aspect = long/short

#     # core type by shape
#     if area_frac_warm >= cfgv("shape.loose_area_frac", 0.10):
#         rule_core = "loose_joint"
#     elif aspect >= cfgv("shape.wire_aspect_min", 2.2) and area_frac_warm < cfgv("shape.wire_area_max", 0.25):
#         rule_core = "wire_overload"
#     else:
#         rule_core = "point_overload"

#     return ok_warm, ok_hot, rule_core, float(area_frac_warm), float(aspect)

# # map (rule_core, is_hot) -> your 5-class names
# def final_name_from_rule(rule_core, is_hot):
#     if rule_core == "wire_overload":
#         return "Full wire overload"                       # always 'potential' per rubric
#     if rule_core == "loose_joint":
#         return "Loose Joint -Faulty" if is_hot else "Loose Joint -potential"
#     if rule_core == "point_overload":
#         # dataset has only 'Point Overload Faulty'; reuse LJ-potential for yellow points
#         return "Point Overload Faulty" if is_hot else "Loose Joint -potential"
#     return None

# # class list for drawing
# CLASS_COLORS = {
#     "Loose Joint -Faulty": (0, 0, 255),
#     "Point Overload Faulty": (0, 128, 255),
#     "Loose Joint -potential": (0, 255, 255),
#     "Full wire overload": (255, 0, 0),
# }

# # --- load model ---
# model = YOLO(WEIGHTS)

# # YOLO sees ORIGINAL colors (as trained)
# cand_raw  = load_rgb(CANDIDATE_IMG)

# # Rules see a NORMALIZED copy
# cand_norm = normalize_palette(cand_raw)

# # Optional baseline for dv_min
# vB = None
# if BASELINE_IMG:
#     base = normalize_palette(load_rgb(BASELINE_IMG))
#     cand_norm = resize_to_match(base, cand_norm)
#     cand_norm = register_affine(base, cand_norm)
#     vB = rgb_to_hsv01(base)[...,2]

# hsvC = rgb_to_hsv01(cand_norm)

# # --- YOLO inference (GPU) ---
# pred = model.predict(cand_raw, device=0, half=True, imgsz=640, verbose=False)[0]
# print(f"YOLO raw detections: {len(pred.boxes)}")

# # --- refine with rules ---
# refined = []
# if len(pred.boxes):
#     xyxy = pred.boxes.xyxy.cpu().numpy()
#     conf = pred.boxes.conf.cpu().numpy()
#     for (x1,y1,x2,y2), cf in zip(xyxy, conf):
#         if float(cf) < cfgv("yolo_min_conf", 0.20):
#             continue
#         x, y, w, h = int(x1), int(y1), int(x2-x1), int(y2-y1)
#         ok_warm, ok_hot, rule_core, areaFrac, aspect = classify_box(hsvC, vB, (x,y,w,h))
#         if not ok_warm:
#             continue
#         final_name = final_name_from_rule(rule_core, ok_hot)
#         if final_name is None or final_name == "normal":
#             continue
#         refined.append({
#             "x": x, "y": y, "w": w, "h": h,
#             "detConfidence": float(cf),
#             "ruleCore": rule_core,
#             "isHot": bool(ok_hot),
#             "finalClass": final_name,
#             "areaFrac": float(areaFrac),
#             "aspect": float(aspect),
#         })

# print(f"After thresholds: {len(refined)} box(es)")

# # --- image-level decision & score ---
# FAULTY = {"Loose Joint -Faulty", "Point Overload Faulty"}
# POT    = {"Loose Joint -potential", "Full wire overload"}

# has_faulty = any(b["finalClass"] in FAULTY for b in refined)
# has_pot    = any(b["finalClass"] in POT for b in refined)
# grade = "faulty" if has_faulty else ("potentially faulty" if has_pot else "normal")

# weights = {**{k:1.0 for k in FAULTY}, **{k:0.7 for k in POT}}
# score = max((weights[b["finalClass"]]*b["detConfidence"] for b in refined), default=0.0)

# print("Image:", CANDIDATE_IMG)
# print("Baseline:", BASELINE_IMG or "(none)")
# print("Final grade:", grade)
# print("Anomaly score:", round(score, 4))
# print("Boxes:")
# for b in refined:
#     print(f"  [{b['finalClass']} | {b['ruleCore']} | hot={b['isHot']}] "
#           f"conf={b['detConfidence']:.3f} xywh=({b['x']},{b['y']},{b['w']},{b['h']}) "
#           f"areaFrac={b['areaFrac']:.4f} aspect={b['aspect']:.2f}")

# # --- save annotated preview ---
# def draw_anno(img_rgb, boxes):
#     out = img_rgb.copy()
#     for b in boxes:
#         x,y,w,h = b["x"], b["y"], b["w"], b["h"]
#         color = CLASS_COLORS.get(b["finalClass"], (200,200,200))
#         cv2.rectangle(out, (x,y), (x+w, y+h), color, 2)
#         label = f"{b['finalClass']} {b['detConfidence']:.2f}"
#         (tw,th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
#         cv2.rectangle(out, (x, y-18), (x+tw+6, y), color, -1)
#         cv2.putText(out, label, (x+3, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1, cv2.LINE_AA)
#     return out

# annot_bgr = draw_anno(cv2.cvtColor(cand_raw, cv2.COLOR_RGB2BGR), refined)
# Path(Path(SAVE_PATH).parent).mkdir(parents=True, exist_ok=True)
# cv2.imwrite(SAVE_PATH, annot_bgr)
# print("Annotated saved to:", SAVE_PATH)


YOLO raw detections: 2
After thresholds: 2 box(es)
Image: C:\Users\Tuf\Downloads\AI Backend\dataset_trans\images\test\T4_faulty_001_png.rf.fdb4096aeac90695ff86252b9734247f.jpg
Baseline: (none)
Final grade: potentially faulty
Anomaly score: 0.3397
Boxes:
  [Full wire overload | wire_overload | hot=True] conf=0.485 xywh=(271,159,34,131) areaFrac=0.0120 aspect=3.85
  [Full wire overload | wire_overload | hot=True] conf=0.473 xywh=(0,113,103,266) areaFrac=0.0025 aspect=2.58
Annotated saved to: runs\detect\preview\annotated_one.png


In [ ]:
# ---- One-cell reusable inference function (YOLO + hot/warm rules + hybrid fallback) ----
from ultralytics import YOLO
from pathlib import Path
import numpy as np, cv2, json
from math import exp

def infer_thermal(
    candidate_img: str,
    weights: str,
    cfg_path: str,
    baseline_img: str | None = None,
    save_annot: str | None = None,
    device: int = 0,
    imgsz: int = 640,
    web_payload: bool = False,
    half: bool = True,
):
    CFG = json.load(open(cfg_path, "r"))

    def cfgv(path, default):
        d = CFG
        for k in path.split("."):
            if not isinstance(d, dict) or k not in d:
                return default
            d = d[k]
        return d

    # --- utils ---
    def load_rgb(path):
        bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if bgr is None: raise FileNotFoundError(path)
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    def normalize_palette(rgb):
        hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
        h,s,v = cv2.split(hsv)
        v = cv2.equalizeHist(v)  # contrast-normalize V only
        return cv2.cvtColor(cv2.merge([h,s,v]), cv2.COLOR_HSV2RGB)

    def resize_to_match(a, b):
        H, W = a.shape[:2]
        return cv2.resize(b, (W, H), interpolation=cv2.INTER_AREA)

    def register_affine(base_rgb, cand_rgb):
        b_gray = cv2.cvtColor(base_rgb, cv2.COLOR_RGB2GRAY)
        c_gray = cv2.cvtColor(cand_rgb, cv2.COLOR_RGB2GRAY)
        warp = np.eye(2,3, dtype=np.float32)
        try:
            _, warp = cv2.findTransformECC(
                b_gray, c_gray, warp, cv2.MOTION_AFFINE,
                (cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT, 100, 1e-6)
            )
            H,W = b_gray.shape
            return cv2.warpAffine(cand_rgb, warp, (W,H), flags=cv2.INTER_LINEAR+cv2.WARP_INVERSE_MAP)
        except cv2.error:
            return cand_rgb

    def rgb_to_hsv01(img_rgb):
        hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[...,0] /= 179.0
        hsv[...,1] /= 255.0
        hsv[...,2] /= 255.0
        return hsv

    # --- per-box classification with warm & hot masks ---
    def classify_box(hsvC, vB, xywh):
        x,y,w,h = map(int, xywh)
        H, W = hsvC.shape[:2]
        x = max(0, min(x, W-1)); y = max(0, min(y, H-1))
        w = max(1, min(w, W-x)); h = max(1, min(h, H-y))

        roi = hsvC[y:y+h, x:x+w]
        hC, sC, vC_roi = roi[...,0], roi[...,1], roi[...,2]

        warm_h = (hC <= 0.17) | (hC >= 0.95)
        warm_s = sC >= cfgv("color.sat_min", 0.35)
        warm_v = vC_roi >= cfgv("color.val_min", 0.50)

        if vB is None:
            vB_roi = np.zeros_like(vC_roi)
        else:
            vB_roi = vB[y:y+h, x:x+w]
        med_base = cv2.medianBlur((vB_roi*255).astype(np.uint8), 31)/255.0
        med_cand = cv2.medianBlur((vC_roi*255).astype(np.uint8), 31)/255.0
        dv = (vC_roi - med_base)
        dl = (vC_roi - med_cand)

        contrast_warm = (dv >= cfgv("color.dv_min", 0.12)) | (dl >= cfgv("color.dl_min", 0.08))
        warm = (warm_h & warm_s & warm_v & contrast_warm)

        # HOT (stricter)
        hot_s = sC >= cfgv("color_hot.sat_min_hot", 0.45)
        hot_v = vC_roi >= cfgv("color_hot.val_min_hot", 0.65)
        hot_contrast = (dv >= cfgv("color_hot.dv_min_hot", 0.18)) | (dl >= cfgv("color.dl_min", 0.08))
        hot = (warm_h & hot_s & hot_v & hot_contrast)

        warm_pix = int(warm.sum())
        hot_pix  = int(hot.sum())

        min_pix = max(4, int(cfgv("shape.min_area_frac", 0.001) * H * W))
        ok_warm = warm_pix >= min_pix
        ok_hot  = hot_pix  >= min_pix

        area_frac_warm = warm_pix / float(H*W)
        short = max(1, min(w,h)); long = max(w,h)
        aspect = long/short

        if area_frac_warm >= cfgv("shape.loose_area_frac", 0.10):
            rule_core = "loose_joint"
        elif aspect >= cfgv("shape.wire_aspect_min", 2.2) and area_frac_warm < cfgv("shape.wire_area_max", 0.25):
            rule_core = "wire_overload"
        else:
            rule_core = "point_overload"

        return ok_warm, ok_hot, rule_core, float(area_frac_warm), float(aspect)

    def final_name_from_rule(rule_core, is_hot):
        if rule_core == "wire_overload":
            return "Full wire overload"
        if rule_core == "loose_joint":
            return "Loose Joint -Faulty" if is_hot else "Loose Joint -potential"
        if rule_core == "point_overload":
            return "Point Overload Faulty" if is_hot else "Loose Joint -potential"
        return "normal"

    # --- global rule probe + box proposal (for fallback) ---
    def global_rule_probe(hsvC, vB):
        hC, sC, vC = hsvC[...,0], hsvC[...,1], hsvC[...,2]
        warm_h = (hC <= 0.17) | (hC >= 0.95)
        warm_s = sC >= cfgv("color.sat_min", 0.35)
        warm_v = vC >= cfgv("color.val_min", 0.50)

        med_cand = cv2.medianBlur((vC*255).astype(np.uint8), 31)/255.0
        if vB is None:
            dl = (vC - med_cand)
            contrast = (dl >= cfgv("color.dl_min", 0.08))
            dv_like = dl
        else:
            med_base = cv2.medianBlur((vB*255).astype(np.uint8), 31)/255.0
            dv = (vC - med_base)
            dl = (vC - med_cand)
            contrast = (dv >= cfgv("color.dv_min", 0.12)) | (dl >= cfgv("color.dl_min", 0.08))
            dv_like = np.maximum(0.0, dv)

        warm = (warm_h & warm_s & warm_v & contrast).astype(np.uint8)
        warm_frac = float(warm.mean())
        dv95 = float(np.quantile(dv_like.ravel(), 0.95))
        score = dv95 + 2.0*warm_frac
        prob = 1.0/(1.0 + exp(-score))
        return float(prob), warm

    def propose_boxes_from_warm(warm_mask):
        H, W = warm_mask.shape
        min_area = max(32, int(cfgv("shape.min_area_frac", 0.001) * H * W))
        k = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
        wm = cv2.morphologyEx(warm_mask, cv2.MORPH_OPEN, k)
        num, labels, stats, _ = cv2.connectedComponentsWithStats(wm, 8)
        boxes = []
        for i in range(1, num):
            x,y,w,h,area = map(int, stats[i])
            if area >= min_area:
                boxes.append((x,y,w,h))
        return boxes

    CLASS_COLORS = {
        "Loose Joint -Faulty": (0, 0, 255),
        "Point Overload Faulty": (0, 128, 255),
        "Loose Joint -potential": (0, 255, 255),
        "Full wire overload": (255, 0, 0),
    }

    # --- Model + preprocessing ---
    model = YOLO(weights)
    cand_raw  = load_rgb(candidate_img)      # YOLO sees original colors
    cand_norm = normalize_palette(cand_raw)  # rules see normalized copy

    vB = None
    if baseline_img:
        base = normalize_palette(load_rgb(baseline_img))
        cand_norm = resize_to_match(base, cand_norm)
        cand_norm = register_affine(base, cand_norm)
        vB = rgb_to_hsv01(base)[...,2]

    hsvC = rgb_to_hsv01(cand_norm)

    # --- YOLO inference ---
    pred = model.predict(cand_raw, device=device, half=half, imgsz=imgsz, verbose=False)[0]

    refined = []
    if len(pred.boxes):
        xyxy = pred.boxes.xyxy.cpu().numpy()
        conf = pred.boxes.conf.cpu().numpy()
        for (x1,y1,x2,y2), cf in zip(xyxy, conf):
            if float(cf) < cfgv("yolo_min_conf", 0.20):
                continue
            x, y, w, h = int(x1), int(y1), int(x2-x1), int(y2-y1)
            ok_warm, ok_hot, rule_core, areaFrac, aspect = classify_box(hsvC, vB, (x,y,w,h))
            if not ok_warm:
                continue
            final_name = final_name_from_rule(rule_core, ok_hot)
            if final_name == "normal":
                continue
            refined.append({
                "x": x, "y": y, "w": w, "h": h,
                "detConfidence": float(cf),
                "ruleCore": rule_core,
                "isHot": bool(ok_hot),
                "finalClass": final_name,
                "areaFrac": float(areaFrac),
                "aspect": float(aspect),
            })

    # --- HYBRID FALLBACK (inside the function!) ---
    rule_prob_thr = cfgv("decision.rule_prob_thr", 0.40)
    use_rule_fallback = cfgv("decision.use_rule_only_fallback", True)
    rule_prob, warm_mask = global_rule_probe(hsvC, vB)

    if (len(refined) == 0) and use_rule_fallback and (rule_prob >= rule_prob_thr):
        for (x, y, w, h) in propose_boxes_from_warm(warm_mask):
            ok_warm, ok_hot, rule_core, areaFrac, aspect = classify_box(hsvC, vB, (x,y,w,h))
            if not ok_warm:
                continue
            if rule_core == "wire_overload":
                final_name = "Full wire overload"
            elif rule_core == "loose_joint":
                final_name = "Loose Joint -Faulty" if ok_hot else "Loose Joint -potential"
            else:
                final_name = "Point Overload Faulty" if ok_hot else "Loose Joint -potential"
            refined.append({
                "x": x, "y": y, "w": w, "h": h,
                "detConfidence": float(rule_prob),
                "ruleCore": rule_core,
                "isHot": bool(ok_hot),
                "finalClass": final_name,
                "areaFrac": float(areaFrac),
                "aspect": float(aspect),
            })

    # --- Image-level decision & score ---
    FAULTY = {"Loose Joint -Faulty", "Point Overload Faulty"}
    POT    = {"Loose Joint -potential", "Full wire overload"}

    has_faulty = any(b["finalClass"] in FAULTY for b in refined)
    has_pot    = any(b["finalClass"] in POT for b in refined)
    grade = "faulty" if has_faulty else ("potentially faulty" if has_pot else "normal")

    weights_for_score = {**{k:1.0 for k in FAULTY}, **{k:0.7 for k in POT}}
    score = max((weights_for_score[b["finalClass"]]*b["detConfidence"] for b in refined), default=0.0)

    full_result = {
        "image": str(candidate_img),
        "grade": grade,
        "anomaly_score": float(score),
        "boxes": refined,
        "rule_prob": float(rule_prob),
    }

    # --- build YOLO-normalized boxes for web ---
    H, W = cand_raw.shape[:2]

    def to_yolo_norm(b):
        x, y, w, h = b["x"], b["y"], b["w"], b["h"]
        cx = (x + w/2.0) / W
        cy = (y + h/2.0) / H
        ww = w / W
        hh = h / H
        # (optional) round for cleaner payloads
        return [round(cx, 6), round(cy, 6), round(ww, 6), round(hh, 6)]

    web_boxes = [
        {
            "class": b["finalClass"],
            "bbox_yolo": to_yolo_norm(b),     # [cx, cy, w, h] normalized
            "bbox_px":   [b["x"], b["y"], b["w"], b["h"]]  # keep pixels too if you want
        }
        for b in refined
    ]


    if web_payload:
        result = {
            "grade": grade,
            "anomaly_score": float(score),
            "boxes": web_boxes
        }
    else:
        result = full_result

    if save_annot:
        out = cv2.cvtColor(cand_raw, cv2.COLOR_RGB2BGR).copy()
        for b in refined:
            x,y,w,h = b["x"], b["y"], b["w"], b["h"]
            color = CLASS_COLORS.get(b["finalClass"], (200,200,200))
            cv2.rectangle(out, (x,y), (x+w, y+h), color, 2)
            label = f"{b['finalClass']} {b['detConfidence']:.2f}"
            (tw,th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(out, (x, y-18), (x+tw+6, y), color, -1)
            cv2.putText(out, label, (x+3, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1, cv2.LINE_AA)
        Path(save_annot).parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(save_annot), out)
        result["annotated"] = str(save_annot)

    return result


In [44]:
# Minimal call from another cell:
res = infer_thermal(
    candidate_img=r"dataset_trans\images\valid\T2_faulty_003_png_png.rf.dc04a72ca738495c80248d579698da0d.jpg",
    baseline_img=None,  # or r"C:\...\some_normal_baseline.jpg"
    weights=r"runs\detect\train\weights\best.pt",
    cfg_path=r"cfg\config_global.json",
    save_annot=r"runs\detect\preview\annot_from_func.png",
    device=0,   # 0=CUDA GPU, -1=CPU
    imgsz=640,
    web_payload=True,
    half=True
)

# print(res["grade"], res["anomaly_score"])
# for b in res["boxes"]:
#     print(b["finalClass"], b["detConfidence"], b["ruleCore"], b["isHot"], b["areaFrac"], b["aspect"])
# print("Annotated:", res.get("annotated"))


print(res["grade"], res["anomaly_score"])
for b in res["boxes"]:
    print(b)


faulty 0.7373046875
{'class': 'Point Overload Faulty', 'bbox_yolo': [0.400591, 0.561258, 0.545276, 0.480132], 'bbox_px': [65, 97, 277, 145]}
